# Análisis Exploratorio de Datos (EDA) - Base de Datos en Bogotá

Este notebook realiza un análisis exploratorio detallado de la base de datos completa de registros de instrumentos públicos en Bogotá.

## Requerimientos Cumplidos:
- Resumen de Estadísticas (Cálculos de tendencia central, dispersión y percentiles).
- Gráficos de Análisis Exploratorio General (Conteo, barras).
- Histogramas y Gráficos de Densidad para conocer la distribución (Aplicado fuertemente al Valor).
- Matriz de Correlaciones de las variables de forma gráfica (Heatmap).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# Configuración de las gráficas
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Crear carpeta para guardar gráficas si no existe
os.makedirs('plots', exist_ok=True)

## 1. Carga de Datos y Limpieza Inicial
Debido al tamaño de la base de datos (~1.18 GB), especificaremos los tipos de datos y cargaremos solo las columnas útiles para reducir significativamente el consumo de RAM. También imputaremos/limpiaremos anomalías básicas en la variable objetivo (`valor`).

In [ ]:
# Definir tipos de datos optimizados para ahorrar RAM
dtypes = {
    'pk': 'str',
    'year_radica': 'float32',
    'orip': 'category',
    'departamento': 'category',
    'tipo_predio_zona': 'category',
    'num_anotacion': 'float32',
    'dinamica_2024': 'float32',
    'cod_natujur': 'category',
    'nombre_natujur': 'category',
    'count_a': 'float32',
    'count_de': 'float32',
    'predios_nuevos': 'float32',
    'tiene_valor': 'float32',
    'tiene_mas_de_un_valor': 'float32',
    'valor': 'object'
}

print("Cargando dataset completo...")
df = pd.read_csv('data/bogota_full_data.csv', dtype=dtypes, 
                 usecols=list(dtypes.keys()) + ['fecha_radica_texto'])
print(f"El dataset tiene {df.shape[0]:,} filas y {df.shape[1]} columnas.")

# Formateo y Limpieza numéica del Valor
df['valor'] = pd.to_numeric(df['valor'], errors='coerce')

# Formatear Fecha (esto consume RAM por lo que se hace luego de leer)
df['fecha_radica_texto'] = pd.to_datetime(df['fecha_radica_texto'], errors='coerce').dt.year

df.head()

## 2. Resumen Estadístico de Variables Numéricas
Generamos la estadística descriptiva para las variables cuantitativas de la base.

In [ ]:
# Seleccionamos variables cuantitativas
numeric_cols = [
    'year_radica', 'num_anotacion', 'dinamica_2024', 
    'count_a', 'count_de', 'predios_nuevos', 'tiene_valor', 'valor'
]

# Mostrar estadisticas y redondear
stats_summary = df[numeric_cols].describe().round(2)
stats_summary

**Explicación del Resumen:**
Observamos la cantidad total de datos válidos (count), los promedios (mean), la desviación (std), además de cuartiles (25%, 50%, 75%) y min/max. Presta especial atención al `valor` donde el máximo de venta será extremadamente grande en comparación al 75%, sugiriendo valores altamente atípicos. Asimismo, algunos valores pueden ser 0, lo cual en bienes raíces debe procesarse con precaución.

## 3. Gráficos de Análisis Exploratorio General (Categóricos)
A continuación, visualizamos el volumen de las variables de clasificación más importantes (Zonificación, Actos Jurídicos, e Histórico de transacciones).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tipos de Zona (Urbano vs Otras)
sns.countplot(data=df, x='tipo_predio_zona', ax=axes[0], palette='viridis')
axes[0].set_title('Distribución por Zona (Urbano vs Rural)')
axes[0].set_xlabel('Zona')
axes[0].set_ylabel('Cantidad de Predios')

# Top 10 Natujur (Naturaleza Jurídica: Compraventas, etc)
top_natujur = df['nombre_natujur'].value_counts().nlargest(10)
sns.barplot(y=top_natujur.index, x=top_natujur.values, ax=axes[1], palette='magma')
axes[1].set_title('Top 10: Actos Jurídicos Más Registrados')
axes[1].set_xlabel('Cantidad')
axes[1].set_ylabel('Naturaleza Jurídica')

plt.tight_layout()
plt.savefig('plots/1_graficos_generales.png', dpi=300)
plt.show()

## 4. Histograma y Gráficos de Densidad (Distribución)
Analizamos la distribución de la variable de interés principal: `valor`. Debido a la colosal asimetría positiva de los inmuebles (hay muchas propiedades normales y muy pocas mansiones de miles de millones), emplearemos Escala Logarítmica e intervalos seguros.

In [ ]:
# Filtrar los valores con sentido para el modelo (Ej. > 1,000,000 para obviar errores de $1)
df_valores = df[df['valor'] > 1e6].copy()

fig, ax = plt.subplots(figsize=(12, 6))

# Histograma con gráfica de densidad (KDE plot)
# Usamos log10 del valor para poder visualizar la campana
log_values = np.log10(df_valores['valor'].dropna())

sns.histplot(log_values, bins=60, kde=True, color="#2ca02c", stat="density", ax=ax)

ax.set_title('Histograma y Curva de Densidad de Probabilidad (KDE) - Valores de Transacción (Log10)')
ax.set_xlabel('Log10 del Valor ($10^6$ = 1 Millón, $10^9$ = 1.000 Millones)')
ax.set_ylabel('Densidad')

plt.savefig('plots/2_histograma_densidad_valores.png', dpi=300)
plt.show()

**Explicación de la Distribución:**
La gráfica confirma visualmente la distribución Log-Normal que siguen comúnmente las bases de datos de inmuebles reales urbanos. La Transformación logarítmica es capaz de empujar los datos y aproximarlos a una curva normal (Campana de Gauss). Este paso es clave para el EDA formal que te pidieron y esencial si vas a modelarlo por MCO (Regresión OLS).

## 5. Matriz de Correlación Gráfica (Heatmap de Todas las Variables)
Para cumplir con el cuarto requerimiento, generamos una matriz completa para ver cómo correlacionan linealmente todos los componentes numéricos.

In [ ]:
# Para que la matriz sea de "todas" las variables, incluiremos algunas categóricas
# convirtiéndolas a dummies matemáticamente (ej. Urbano=1, Rural=0)
df_corr = df[numeric_cols].copy()

# Variable Dummy para observar si la propiedad es URBANO
df_corr['es_urbano'] = (df['tipo_predio_zona'] == 'URBANO').astype(int)

# Variable Dummy temporal para observar si el predio fue COMPRAVENTA
df_corr['es_compraventa'] = (df['nombre_natujur'] == 'COMPRAVENTA').astype(int)

# Calcular matriz Pearson (se omiten nulos con .corr)
corr_matrix = df_corr.corr(method='pearson')

# Dibujar 
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # Para tapar el espejo superior
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"shrink": .8})

plt.title('Matriz de Correlación de Todas las Variables', fontsize=15)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('plots/3_matriz_correlacion.png', dpi=300)
plt.show()

**Explicación:**
En este Heatmap visualizamos el coeficiente de asosciación que va desde -1 (relación negativa muy fuerte) hasta rojo intenso de 1 (relación directa y rígida). Por ejemplo: Podemos chequear correlaciones entre el hecho de que posea valor con la variable Compraventa. El Heatmap es excelente para descartar la inclusión de dos variables que midan lo mismo, y evitar problemas de multicolinealidad estructural al pasar después a los algortimos predictivos.